# 01 - Data Ingestion & Exploration
## Italian Electricity Grid Data from Terna

This notebook loads data from Terna (Italian TSO) and performs initial exploration.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.terna_loader import load_demand_data, load_generation_data, load_frequency_data, merge_demand_generation

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## Load Data

In [ ]:
demand = load_demand_data()
print(f'Demand shape: {demand.shape}')
print(f'Date range: {demand.index.min()} to {demand.index.max()}')
demand.head(10)

In [ ]:
generation = load_generation_data()
print(f'Generation shape: {generation.shape}')
generation.head(10)

In [ ]:
frequency = load_frequency_data()
print(f'Frequency shape: {frequency.shape}')
frequency.head(10)

## Demand Time Series

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(demand.index, demand['demand_mw'], linewidth=0.5, alpha=0.7)
axes[0].set_title('Italian Electricity Demand - Full Series')
axes[0].set_ylabel('Demand (MW)')

weekly = demand['demand_mw'].resample('W').mean()
axes[1].plot(weekly.index, weekly, linewidth=1.5, color='orange')
axes[1].set_title('Weekly Average Demand')
axes[1].set_ylabel('Demand (MW)')

plt.tight_layout()
plt.show()

## Generation Mix

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flatten(), generation.columns):
    daily = generation[col].resample('D').mean()
    ax.plot(daily.index, daily, linewidth=0.8)
    ax.set_title(col.replace('_mw', '').title())
    ax.set_ylabel('MW')

plt.suptitle('Generation by Source - Daily Average', fontsize=14)
plt.tight_layout()
plt.show()

## Statistical Summary

In [ ]:
print('=== Demand Statistics ===')
print(demand['demand_mw'].describe())
print(f'\nSkewness: {demand["demand_mw"].skew():.3f}')
print(f'Kurtosis: {demand["demand_mw"].kurtosis():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(demand['demand_mw'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('Demand (MW)')

from scipy import stats
stats.probplot(demand['demand_mw'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

## Hourly and Daily Patterns

In [ ]:
demand_temp = demand.copy()
demand_temp['hour'] = demand_temp.index.hour
demand_temp['dayofweek'] = demand_temp.index.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_avg = demand_temp.groupby('hour')['demand_mw'].mean()
axes[0].plot(hourly_avg.index, hourly_avg.values, marker='o')
axes[0].set_title('Average Demand by Hour')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Demand (MW)')
axes[0].set_xticks(range(0, 24, 2))

daily_avg = demand_temp.groupby('dayofweek')['demand_mw'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1].bar(range(7), daily_avg.values, tick_label=days)
axes[1].set_title('Average Demand by Day of Week')
axes[1].set_ylabel('Demand (MW)')

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
merged = merge_demand_generation()
print(f'Merged shape: {merged.shape}')

corr = merged.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Frequency Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(frequency.index[:500], frequency['frequency_hz'].iloc[:500], linewidth=0.5)
axes[0].axhline(y=50.0, color='r', linestyle='--', label='Nominal 50 Hz')
axes[0].set_title('Grid Frequency (first 500 hours)')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].legend()

axes[1].hist(frequency['frequency_hz'], bins=100, edgecolor='black', alpha=0.7)
axes[1].axvline(x=50.0, color='r', linestyle='--', label='Nominal 50 Hz')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Notebook 01 complete. Data loaded and explored.')